In [ ]:
# Databricks notebook source
# tutorial_name: 03-Day8-Medallion-Bronze-Task
# notebookName: 03-Day8-Medallion-Bronze-Task
# COMMAND ----------

# Databricks notebook source
# COMMAND ----------

# DBTITLE 1,Paths (same as Days 5–8)
notepoint_rel = "hands-on/day-08/notebooks/03-Day8-Medallion-Bronze-Task.ipynb"
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u25"  # u01–u16
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
SOURCE_CSV = BASE_PATH + "/2010-summary.csv"
SOURCE_JSON = BASE_PATH + "/json/2015-summary.json"
DAY8_ROOT = f"{BASE_PATH}day08-{STUDENT_ID}"
MED_ROOT = f"{DAY8_ROOT}/medallion"
BRONZE_PATH = f"{MED_ROOT}/bronze_flight_routes"
SILVER_PATH = f"{MED_ROOT}/silver_flight_routes"
GOLD_PATH = f"{MED_ROOT}/gold_by_destination"
METRICS_PATH = f"{DAY8_ROOT}/workflow_metrics/run_metrics"
# COMMAND ----------
print("DAY8_ROOT:", DAY8_ROOT)
print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print("OK: P_BASIC")
except Exception as e:
    print("P_BASIC:", e)
try:
    spark.read.format("csv").option("header", True).load(SOURCE_CSV).limit(1).collect()
    print("OK: SOURCE_CSV")
except Exception as e:
    print("SOURCE_CSV:", e)
try:
    spark.read.format("json").load(SOURCE_JSON).limit(1).collect()
    print("OK: SOURCE_JSON (optional)")
except Exception as e:
    print("(optional) SOURCE_JSON:", type(e).__name__)
# COMMAND ----------


## Bronze task (item 22)

**Purpose:** Read **`SOURCE_CSV`** and write **Delta** to **`BRONZE_PATH`** with an **`ingest_ts`** column.

**When to run:** As **task 1** in the multi-task job, or interactively while learning.

**Output:** Overwrites bronze each run (`mode("overwrite")`).


## Inspect source CSV (before write)

In [ ]:
src_preview = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(SOURCE_CSV)
)
print("Columns:", src_preview.columns)
print("Row count:", src_preview.count())
src_preview.show(5, truncate=False)


In [ ]:
_cols = src_preview.dtypes
print("Bronze will store", len(_cols), "columns:")
for name, typ in _cols:
    print(f"  - {name}: {typ}")


## Write Bronze Delta

In [ ]:
from pyspark.sql import functions as F

df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(SOURCE_CSV)
)
df = df.withColumn("ingest_ts", F.current_timestamp())
df.write.format("delta").mode("overwrite").save(BRONZE_PATH)
print("Bronze rows:", df.count(), "->", BRONZE_PATH)
spark.read.format("delta").load(BRONZE_PATH).limit(5).show(truncate=False)
# COMMAND ----------
